In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

customers = spark.createDataFrame([
    (1,"John ","Hyderabad"),
    (2,"Alice","Chennai"),
    (3,None,"Bangalore")
], ["customer_id","name","city"])

customers.show()

+-----------+-----+---------+
|customer_id| name|     city|
+-----------+-----+---------+
|          1|John |Hyderabad|
|          2|Alice|  Chennai|
|          3| NULL|Bangalore|
+-----------+-----+---------+



In [0]:
cars = spark.createDataFrame([
    (101,"Toyota","Camry",30000),
    (102,"Honda","Civic",-20000),
    (103,"Hyundai","i20",15000)
],["car_id","brand","model","price"])

sales = spark.createDataFrame([
    (1,1,101,"2024-01-01",1),
    (2,2,102,"2024-01-02",2),
    (3,99,103,"2024-01-03",1)
],["sale_id","customer_id","car_id","sale_date","quantity"])
cars.show()
sales.show()

+------+-------+-----+------+
|car_id|  brand|model| price|
+------+-------+-----+------+
|   101| Toyota|Camry| 30000|
|   102|  Honda|Civic|-20000|
|   103|Hyundai|  i20| 15000|
+------+-------+-----+------+

+-------+-----------+------+----------+--------+
|sale_id|customer_id|car_id| sale_date|quantity|
+-------+-----------+------+----------+--------+
|      1|          1|   101|2024-01-01|       1|
|      2|          2|   102|2024-01-02|       2|
|      3|         99|   103|2024-01-03|       1|
+-------+-----------+------+----------+--------+



###Identifying Null values in customers.name

In [0]:
from pyspark.sql.functions import col
names=customers.filter(customers["name"].isNull()).show()


+-----------+----+---------+
|customer_id|name|     city|
+-----------+----+---------+
|          3|NULL|Bangalore|
+-----------+----+---------+



###Negative values present in cars.price

In [0]:
from pyspark.sql.functions import col
prices=cars.filter(cars["price"]<0).show()

+------+-----+-----+------+
|car_id|brand|model| price|
+------+-----+-----+------+
|   102|Honda|Civic|-20000|
+------+-----+-----+------+



###Identifying mismatch values 

In [0]:
mismatched_customers=sales.join(customers,"customer_id","left_anti").show()

+-----------+-------+------+----------+--------+
|customer_id|sale_id|car_id| sale_date|quantity|
+-----------+-------+------+----------+--------+
|         99|      3|   103|2024-01-03|       1|
+-----------+-------+------+----------+--------+



In [0]:
mismatched_cars=sales.join(cars,"car_id","left_anti").show()

+------+-------+-----------+---------+--------+
|car_id|sale_id|customer_id|sale_date|quantity|
+------+-------+-----------+---------+--------+
+------+-------+-----------+---------+--------+



###Multiple records per customer due to sales table

In [0]:
sales.groupby("customer_id").count().show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
|          1|    1|
|          2|    1|
|         99|    1|
+-----------+-----+



###Phase 2 – Cleaning
• Null names handled or documented


In [0]:
cleaned_data=customers.fillna({"name":"unknown"}).show()

+-----------+-------+---------+
|customer_id|   name|     city|
+-----------+-------+---------+
|          1|  John |Hyderabad|
|          2|  Alice|  Chennai|
|          3|unknown|Bangalore|
+-----------+-------+---------+



###negative prices removed

In [0]:
from pyspark.sql.functions import *
neg_prices=cars.withColumn("price",abs(col("price"))).show()

+------+-------+-----+-----+
|car_id|  brand|model|price|
+------+-------+-----+-----+
|   101| Toyota|Camry|30000|
|   102|  Honda|Civic|20000|
|   103|Hyundai|  i20|15000|
+------+-------+-----+-----+



###Trim strings

In [0]:
from pyspark.sql.functions import *
trim_string=customers.withColumn("name",trim(col("name"))).show()

+-----------+-----+---------+
|customer_id| name|     city|
+-----------+-----+---------+
|          1| John|Hyderabad|
|          2|Alice|  Chennai|
|          3| NULL|Bangalore|
+-----------+-----+---------+



In [0]:
print(spark)

###Identify Invalid foreign keys 

In [0]:
customers_invalid=sales.join(customers,"customer_id","left_anti").show()


+-----------+-------+------+----------+--------+
|customer_id|sale_id|car_id| sale_date|quantity|
+-----------+-------+------+----------+--------+
|         99|      3|   103|2024-01-03|       1|
+-----------+-------+------+----------+--------+



In [0]:
cars_invalid=sales.join(cars,"car_id","left_anti").show()


+------+-------+-----------+---------+--------+
|car_id|sale_id|customer_id|sale_date|quantity|
+------+-------+-----------+---------+--------+
+------+-------+-----------+---------+--------+



###create validation report:

In [0]:
print("customers_data:",customers.count())
print("cars_data:",cars.count())
print("sales_data:",sales.count())

customers_data: 3
cars_data: 3
sales_data: 3


In [0]:
customers.filter(col("name").isNull()).count()

1

In [0]:
cars.filter(col("price") < 0).count()

1

In [0]:
invalid_customers = sales.join(customers, "customer_id", "left_anti")
invalid_customers.count()

1

In [0]:
invalid_cars = sales.join(cars, "car_id", "left_anti")
invalid_cars.count()

0

###PHASE 4 – TRANSFORMATION

In [0]:
from pyspark.sql.functions import *

# Step 1: Trim names
customers_clean = customers.withColumn("name", trim(col("name")))

# Step 2: Handle nulls
customers_clean = customers_clean.fillna({"name": "Unknown"})

# Step 3: Remove negative prices
cars_clean = cars.filter(col("price") > 0)

# Step 4: Identify invalid records
invalid_customers = sales.join(customers_clean, "customer_id", "left_anti")
invalid_cars = sales.join(cars_clean, "car_id", "left_anti")

# Step 5: Remove invalid records
sales_clean = sales.join(customers_clean, "customer_id", "inner") \
                   .join(cars_clean, "car_id", "inner")

# Final cleaned dataset
sales_clean.show()

+------+-----------+-------+----------+--------+----+---------+------+-----+-----+
|car_id|customer_id|sale_id| sale_date|quantity|name|     city| brand|model|price|
+------+-----------+-------+----------+--------+----+---------+------+-----+-----+
|   101|          1|      1|2024-01-01|       1|John|Hyderabad|Toyota|Camry|30000|
+------+-----------+-------+----------+--------+----+---------+------+-----+-----+



In [0]:
df = sales_clean.withColumn("revenue", col("price") * col("quantity"))

In [0]:
customer_revenue = df.groupBy("customer_id") \
    .agg(sum("revenue").alias("total_revenue"))

customer_revenue.show()

+-----------+-------------+
|customer_id|total_revenue|
+-----------+-------------+
|          1|        30000|
+-----------+-------------+



In [0]:
brand_sales = df.groupBy("brand") \
    .agg(sum("revenue").alias("brand_revenue"))

brand_sales.show()

+------+-------------+
| brand|brand_revenue|
+------+-------------+
|Toyota|        30000|
+------+-------------+



In [0]:
city_revenue = df.groupBy("city") \
    .agg(sum("revenue").alias("city_revenue"))

city_revenue.show()

+---------+------------+
|     city|city_revenue|
+---------+------------+
|Hyderabad|       30000|
+---------+------------+



###PHASE 5 – DEALER ANALYTICS

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

df = df.withColumn("dealer_id", (col("customer_id") % 2) + 1)

In [0]:
dealer_revenue = df.groupBy("dealer_id") \
    .agg(sum("revenue").alias("dealer_revenue"))

dealer_revenue.show()

+---------+--------------+
|dealer_id|dealer_revenue|
+---------+--------------+
|        2|         30000|
+---------+--------------+



In [0]:
top_dealers = dealer_revenue.orderBy(col("dealer_revenue").desc())

top_dealers.show()

+---------+--------------+
|dealer_id|dealer_revenue|
+---------+--------------+
|        2|         30000|
+---------+--------------+



In [0]:
dealer_city = df.groupBy("dealer_id", "city") \
    .agg(sum("revenue").alias("revenue"))

dealer_city.show()

+---------+---------+-------+
|dealer_id|     city|revenue|
+---------+---------+-------+
|        2|Hyderabad|  30000|
+---------+---------+-------+



###PHASE 6 – SQL ANALYSIS

In [0]:
df.createOrReplaceTempView("sales_data")

In [0]:
spark.sql("""
SELECT *
FROM (
    SELECT customer_id, city, SUM(revenue) AS total_revenue,
           RANK() OVER (PARTITION BY city ORDER BY SUM(revenue) DESC) as rank
    FROM sales_data
    GROUP BY customer_id, city
) t
WHERE rank <= 3
""").show()

+-----------+---------+-------------+----+
|customer_id|     city|total_revenue|rank|
+-----------+---------+-------------+----+
|          1|Hyderabad|        30000|   1|
+-----------+---------+-------------+----+



In [0]:
spark.sql("""
SELECT date_format(sale_date, 'yyyy-MM') as month,
       SUM(revenue) as total_revenue
FROM sales_data
GROUP BY month
ORDER BY month
""").show()

+-------+-------------+
|  month|total_revenue|
+-------+-------------+
|2024-01|        30000|
+-------+-------------+



In [0]:
spark.sql("""
SELECT customer_id, COUNT(*) as purchase_count
FROM sales_data
GROUP BY customer_id
HAVING COUNT(*) > 1
""").show()

+-----------+--------------+
|customer_id|purchase_count|
+-----------+--------------+
+-----------+--------------+



###PHASE 7 – OUTPUT

In [0]:
customer_revenue.write.mode("overwrite").saveAsTable("customer_revenue_table")